In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

In [15]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') # Added to resolve LookupError

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

# Build a Text Cleaning Pipeline

In [9]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
  """
  This function cleans and preprocesses text data.
  """
  # Convert the input to small/lower order.
  data = dataset.lower()

  # Remove URLs
  data = re.sub(r'https?://\S+|www\.\S+', '', data)

  # Remove mentions (@username)
  data = re.sub(r'@\w+', '', data)

  # Remove emojis (a basic approach, more comprehensive regex might be needed for all emojis)
  emoji_pattern = re.compile("[" # Start of character set
                       "\U0001F600-\U0001F64F"  # emoticons
                       "\U0001F300-\U0001F5FF"  # symbols & pictographs
                       "\U0001F680-\U0001F6FF"  # transport & map symbols
                       "\U0001F1E0-\U0001F1FF"  # flags (iOS)
                       "\U00002702-\U000027B0"
                       "\U000024C2-\U0001F251"
                       "]+", flags=re.UNICODE)
  data = emoji_pattern.sub(r'', data)

  # Remove all other unwanted characters (punctuation and numbers).
  # Removing punctuation first using string.punctuation
  data = data.translate(str.maketrans('', '', string.punctuation))
  # Then remove numbers
  data = re.sub(r'\d+', '', data)

  # Create tokens.
  tokens = word_tokenize(data)

  # Remove stopwords:
  stop_words = set(stopwords.words('english'))
  tokens = [word for word in tokens if word not in stop_words]

  if rule == "lemmatize":
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
  else:
    print("Invalid rule. Pick between 'lemmatize' or 'stem'")
    return ""

  return " ".join(tokens)

In [11]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
  """
  This function cleans and preprocesses text data.
  """
  # Convert the input to small/lower order.
  data = dataset.lower()

  # Remove URLs
  data = re.sub(r'https?://\S+|www\.\S+', '', data)

  # Remove mentions (@username)
  data = re.sub(r'@\w+', '', data)

  # Remove emojis (a basic approach, more comprehensive regex might be needed for all emojis)
  emoji_pattern = re.compile("[" # Start of character set
                       "\U0001F600-\U0001F64F"  # emoticons
                       "\U0001F300-\U0001F5FF"  # symbols & pictographs
                       "\U0001F680-\U0001F6FF"  # transport & map symbols
                       "\U0001F1E0-\U0001F1FF"  # flags (iOS)
                       "\U00002702-\U000027B0"
                       "\U000024C2-\U0001F251"
                       "]+", flags=re.UNICODE)
  data = emoji_pattern.sub(r'', data)

  # Remove all other unwanted characters (punctuation and numbers).
  # Removing punctuation first using string.punctuation
  data = data.translate(str.maketrans('', '', string.punctuation))
  # Then remove numbers
  data = re.sub(r'\d+', '', data)

  # Create tokens.
  tokens = word_tokenize(data)

  # Remove stopwords:
  stop_words = set(stopwords.words('english'))
  tokens = [word for word in tokens if word not in stop_words]

  if rule == "lemmatize":
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
  else:
    print("Invalid rule. Pick between 'lemmatize' or 'stem'")
    return ""

  return " ".join(tokens)

In [12]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/AI and Machine Learning/trum_tweet_sentiment_analysis.csv')

# Display the first few rows and the column names to verify
print(df.head())
print(df.columns)

                                                text  Sentiment
0  RT @JohnLeguizamo: #trump not draining swamp b...          0
1  ICYMI: Hackers Rig FM Radio Stations To Play A...          0
2  Trump protests: LGBTQ rally in New York https:...          1
3  "Hi I'm Piers Morgan. David Beckham is awful b...          0
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...          0
Index(['text', 'Sentiment'], dtype='object')


### Applying Text Cleaning and Tokenization

In [16]:
# Apply the text cleaning pipeline to the 'text' column
df['cleaned_text'] = df['text'].apply(text_cleaning_pipeline)

# Display the DataFrame with the new cleaned_text column
display(df[['text', 'cleaned_text', 'Sentiment']].head())

,text,cleaned_text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,rt trump draining swamp taxpayer dollar trip a...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,icymi hacker rig fm radio station play antitru...,0
2,Trump protests: LGBTQ rally in New York https:...,trump protest lgbtq rally new york bbcworld via,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",hi im pier morgan david beckham awful donald t...,0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,rt tech firm suing buzzfeed publishing unverif...,0


### Train-Test Split

In [17]:
from sklearn.model_selection import train_test_split

X = df['cleaned_text'] # Features are the cleaned text
y = df['Sentiment']    # Target variable is 'Sentiment'

# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Display the shapes of the resulting datasets
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (1480098,)
X_test shape: (370025,)
y_train shape: (1480098,)
y_test shape: (370025,)


### TF-IDF Vectorization

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to 5000 for practicality

# Fit and transform the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform the testing data
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"X_train_tfidf shape: {X_train_tfidf.shape}")
print(f"X_test_tfidf shape: {X_test_tfidf.shape}")

X_train_tfidf shape: (1480098, 5000)
X_test_tfidf shape: (370025, 5000)


### Model Training and Evaluation

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Initialize Logistic Regression model
logistic_model = LogisticRegression(max_iter=1000) # Increased max_iter for convergence

# Train the model
logistic_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = logistic_model.predict(X_test_tfidf)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248842
           1       0.90      0.87      0.89    121183

    accuracy                           0.93    370025
   macro avg       0.92      0.91      0.92    370025
weighted avg       0.93      0.93      0.93    370025



# Text Classification using Machine Learning Models


### Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.
